# BPIC-17 Analysis — 3.3 Notebook


In [1]:
import pm4py
import pandas as pd
import numpy as np
from IPython.display import display

CASE_COL = "case:concept:name"
ACT_COL  = "concept:name"
TS_COL   = "time:timestamp"

log_raw = pm4py.read_xes("BPIC2017.xes")
log_raw = pm4py.format_dataframe(
    log_raw,
    case_id=CASE_COL,
    activity_key=ACT_COL,
    timestamp_key=TS_COL,
)
print(f"log_raw: {log_raw[CASE_COL].nunique():,} cases  |  {len(log_raw):,} events")

/Users/cole/.local/share/virtualenvs/BPM-lrkv0Mze/lib/python3.14/site-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

log_raw: 31,509 cases  |  1,202,267 events


# Helper Functions

In [2]:
def compute_variants(df):
    return (
        df.sort_values(TS_COL)
        .groupby(CASE_COL)[ACT_COL]
        .apply(tuple)
        .nunique()
    )

def case_length_stats(df):
    lengths = df.groupby(CASE_COL).size()
    return lengths.mean(), lengths.std()

def case_duration_stats(df):
    grp = df.groupby(CASE_COL)[TS_COL]
    secs = (grp.max() - grp.min()).dt.total_seconds()
    return secs.mean(), secs.std()


## Filtering Pipeline


In [3]:
log_c = log_raw[log_raw["lifecycle:transition"] == "complete"].copy()

print(f"Stage 1 — filter to complete events")
print(f"  Cases:  {log_raw[CASE_COL].nunique():,} → {log_c[CASE_COL].nunique():,}  (unchanged)")
print(f"  Events: {len(log_raw):,} → {len(log_c):,}  ({len(log_raw)-len(log_c):,} lifecycle events removed)")

Stage 1 — filter to complete events
  Cases:  31,509 → 31,509  (unchanged)
  Events: 1,202,267 → 475,306  (726,961 lifecycle events removed)


In [4]:
OUTCOMES = {"A_Pending", "A_Cancelled", "A_Denied"}
outcome_cases  = log_c[log_c[ACT_COL].isin(OUTCOMES)][CASE_COL].unique()
log_c_outcome = log_c[log_c[CASE_COL].isin(outcome_cases)].copy()

outcome_label = (
    log_c_outcome[log_c_outcome[ACT_COL].isin(OUTCOMES)]
    .groupby(CASE_COL)[ACT_COL]
    .first()
    .rename("outcome")
)
log_c_outcome = log_c_outcome.merge(outcome_label, on=CASE_COL, how="left")
log_outcomes_original = log_c_outcome


print(f"\nStage 2 — log_c_outcome (business outcomes, PRIMARY)")
print(f"  Cases:  {log_c[CASE_COL].nunique():,} → {log_c_outcome[CASE_COL].nunique():,}")
print(f"  Events: {len(log_c):,} → {len(log_c_outcome):,}")
print(f"\n  Outcome distribution:")
print(log_c_outcome.drop_duplicates(CASE_COL)["outcome"].value_counts().to_string())


Stage 2 — log_c_outcome (business outcomes, PRIMARY)
  Cases:  31,509 → 31,411
  Events: 475,306 → 473,963

  Outcome distribution:
outcome
A_Pending      17228
A_Cancelled    10431
A_Denied        3752


In [5]:
n_variants_c = compute_variants(log_c_outcome)

variant_counts = (
    log_c.sort_values(TS_COL)
    .groupby(CASE_COL)[ACT_COL]
    .apply(tuple)
    .value_counts()
)
cumulative = variant_counts.cumsum() / variant_counts.sum()
coverage_rank = {}
for t in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95, 0.99]:
    arr = (cumulative >= t).values
    coverage_rank[t] = int(arr.argmax()) + 1 if arr.any() else len(cumulative)

case_variants = log_c.sort_values(TS_COL).groupby(CASE_COL)[ACT_COL].apply(tuple)

denied_variants = variant_counts[variant_counts.index.map(lambda v: "A_Denied" in v)]
top_denied      = {denied_variants.index[0], denied_variants.index[2]}
n_total_c       = log_c[CASE_COL].nunique()


top10_variants_set   = set(variant_counts.iloc[:10].index)
top10_cases          = case_variants[case_variants.isin(top10_variants_set)].index
log_top30            = log_c[log_c[CASE_COL].isin(top10_cases)].copy()

denied_variant_cases = case_variants[case_variants.isin(top_denied)].index
top30_denied_cases   = top10_cases.union(denied_variant_cases)
log_top30_denied     = log_c[log_c[CASE_COL].isin(top30_denied_cases)].copy()


# ── OUTPUT ──────────────────────────────────────────────────────────────────
print("=" * 85)
print("§3.2 COMPLEMENT & STAGE 3 ANALYSIS")
print("=" * 85)
print(f"  Process variants (log_c): {n_variants_c:,}")
print()

# Variant Coverage Table
print("  Variant coverage (log_c cases):")
print(f"  {'Cov':>3}  {'Top-N':>6}  {'Cases':>7}  {'A_Pending':>12}  {'A_Cancelled':>12}  {'A_Denied':>10}")
print("  " + "-" * 75)
for t, n in coverage_rank.items():
    top_vars  = set(variant_counts.iloc[:n].index)
    top_cases_idx = case_variants[case_variants.isin(top_vars)].index
    total     = len(top_cases_idx)
    dist      = outcome_label.reindex(top_cases_idx).value_counts()
    print(f"  {int(t*100):>3d}%  top {n:>5,}  {total:>7,}  "
          f"{dist.get('A_Pending', 0):>12,}  {dist.get('A_Cancelled', 0):>12,}  {dist.get('A_Denied', 0):>10,}")


§3.2 COMPLEMENT & STAGE 3 ANALYSIS
  Process variants (log_c): 5,562

  Variant coverage (log_c cases):
  Cov   Top-N    Cases     A_Pending   A_Cancelled    A_Denied
  ---------------------------------------------------------------------------
   20%  top     4    6,673           969         5,704           0
   30%  top    10    9,817         3,195         6,622           0
   40%  top    18   12,823         5,579         7,244           0
   50%  top    35   15,865         6,733         7,943       1,189
   60%  top    73   18,918         9,169         8,129       1,620
   70%  top   188   22,057        11,050         8,790       2,217
   80%  top   610   25,209        13,028         9,438       2,713
   85%  top 1,185   26,783        14,050         9,724       2,969
   90%  top 2,473   28,359        15,100         9,960       3,241
   95%  top 4,048   29,934        16,149        10,220       3,490
   99%  top 5,308   31,194        17,022        10,385       3,693


## Change Over Filtering


In [6]:
def funnel_row(label, df):
    n_c = df[CASE_COL].nunique()
    n_e = len(df)
    lm, ls = case_length_stats(df)
    dm, ds = case_duration_stats(df)
    return {
        "Log":             label,
        "Cases":           f"{n_c:,}",
        "Events":          f"{n_e:,}",
        "% of raw cases":  f"{n_c/31509*100:.1f}%",
        "Mean len (evts)": f"{lm:.1f}±{ls:.1f}",
        "Dur (days)": f"{dm/86400:.1f}±{ds/86400:.1f}",
    }

funnel = pd.DataFrame([
    funnel_row("log_raw          (all lifecycle events)",    log_raw),
    funnel_row("log_c            (complete events only)",    log_c),
    funnel_row("log_c_outcome    (3 business outcomes)",     log_c_outcome),
    funnel_row("log_top30        (top 10 variants, 30%)",    log_top30),
    funnel_row("log_top30_denied (log_top30 + 1st/3rd denied)", log_top30_denied),
]).set_index("Log")

print("=== Filtering Funnel ===")
display(funnel)

=== Filtering Funnel ===


,Cases,Events,% of raw cases,Mean len (evts),Dur (days)
Log,,,,,
log_raw (all lifecycle events),"31,509","1,202,267",100.0%,38.2±16.7,21.9±13.2
log_c (complete events only),"31,509","475,306",100.0%,15.1±4.6,21.8±12.9
log_c_outcome (3 business outcomes),"31,411","473,963",99.7%,15.1±4.6,21.8±12.9
"log_top30 (top 10 variants, 30%)","9,817","110,124",31.2%,11.2±1.2,24.1±11.2
log_top30_denied (log_top30 + 1st/3rd denied),"10,254","115,580",32.5%,11.3±1.2,23.7±11.2


## §3.3 Process Model Discovery

In [7]:
print("Discovering model...")

bpmn_im02_c = pm4py.discover_bpmn_inductive(log_c_outcome, noise_threshold=0.2)
bpmn_im03_c = pm4py.discover_bpmn_inductive(log_c_outcome, noise_threshold=0.3)
bpmn_im04_c = pm4py.discover_bpmn_inductive(log_c_outcome, noise_threshold=0.4)

bpmn_im02_denied = pm4py.discover_bpmn_inductive(log_top30_denied, noise_threshold=0.2)
bpmn_im03_denied = pm4py.discover_bpmn_inductive(log_top30_denied, noise_threshold=0.3)

net_im02_c, im_im02_c, fm_im02_c = pm4py.convert_to_petri_net(bpmn_im02_c)
print("  IM noise=0.2 (log_c_outcome) done")

net_im03_c, im_im03_c, fm_im03_c = pm4py.convert_to_petri_net(bpmn_im03_c)
print("  IM noise=0.3 (log_c_outcome) done")

net_im04_c, im_im04_c, fm_im04_c = pm4py.convert_to_petri_net(bpmn_im04_c)
print("  IM noise=0.4 (log_c_outcome) done")

net_im02_denied, im_im02_denied, fm_im02_denied = pm4py.convert_to_petri_net(bpmn_im02_denied)
print("  IM noise=0.2 (log_top30_denied) done")

net_im03_denied, im_im03_denied, fm_im03_denied = pm4py.convert_to_petri_net(bpmn_im03_denied)
print("  IM noise=0.3 (log_top30_denied) done")

net_hm02, im_hm02, fm_hm02 = pm4py.discover_petri_net_heuristics(log_top30_denied, dependency_threshold=0.2)
print("  HM done")

models = {
    "IM (noise=0.2, log_c_outcome)   ": (net_im02_c,      im_im02_c,      fm_im02_c),
    "IM (noise=0.3, log_c_outcome)   ": (net_im03_c,      im_im03_c,      fm_im03_c),
    "IM (noise=0.4, log_c_outcome)   ": (net_im04_c,      im_im04_c,      fm_im04_c),
    "IM (noise=0.2, log_top30_denied)": (net_im02_denied, im_im02_denied, fm_im02_denied),
    "IM (noise=0.3, log_top30_denied)": (net_im03_denied, im_im03_denied, fm_im03_denied),
    "HM (dependency=0.2, log_top30_denied)": (net_hm02, im_hm02, fm_hm02),
}

print()
print(f"{'Model':<35s}  {'places':>7s}  {'transitions':>12s}  {'arcs':>6s}")
print("-" * 65)
for name, (net, im, fm) in models.items():
    print(f"{name:<35s}  {len(net.places):>7d}  {len(net.transitions):>12d}  {len(net.arcs):>6d}")

Discovering model...
  IM noise=0.2 (log_c_outcome) done
  IM noise=0.3 (log_c_outcome) done
  IM noise=0.4 (log_c_outcome) done
  IM noise=0.2 (log_top30_denied) done
  IM noise=0.3 (log_top30_denied) done
  HM done

Model                                 places   transitions    arcs
-----------------------------------------------------------------
IM (noise=0.2, log_c_outcome)             37            56     116
IM (noise=0.3, log_c_outcome)             35            54     112
IM (noise=0.4, log_c_outcome)             28            45      94
IM (noise=0.2, log_top30_denied)          25            30      62
IM (noise=0.3, log_top30_denied)          22            25      52
HM (dependency=0.2, log_top30_denied)       21            25      53


In [8]:
# ── Custom simplicity metrics ─────────────────────────────────────────────────
# Size: total node count (places + transitions). Fewer nodes → simpler model.
def metric_size(net):
    return len(net.places) + len(net.transitions)

# CFC (Control-Flow Complexity): sum of out-degree at every split node (out-degree > 1).
# Captures branching complexity — higher CFC means more decision points.
def metric_cfc(net):
    out_deg = {}
    for arc in net.arcs:
        out_deg[arc.source] = out_deg.get(arc.source, 0) + 1
    return sum(d for d in out_deg.values() if d > 1)

metric_rows = {}
for name, (net, im, fm) in models.items():
    print(f"Evaluating {name} ...", end=" ", flush=True)
    fit  = pm4py.fitness_token_based_replay(log_outcomes_original, net, im, fm)
    prec = pm4py.precision_token_based_replay(log_outcomes_original, net, im, fm)
    gen  = pm4py.generalization_tbr(log_outcomes_original, net, im, fm)
    simp = pm4py.simplicity_petri_net(net, im, fm)
    metric_rows[name] = {
        "fitness":          round(fit["log_fitness"], 4),
        "precision":        round(prec, 4),
        "generalization":   round(gen, 4),
        "simplicity_pm4py": round(simp, 4),
        "size":             metric_size(net),
        "cfc":              metric_cfc(net),
    }
    print("done")

df_metrics = pd.DataFrame(metric_rows).T
print("\n=== Quality Metrics (log_c_outcome) ===")
display(df_metrics)


Evaluating IM (noise=0.2, log_c_outcome)    ... 

replaying log with TBR, completed traces ::   0%|          | 0/5562 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/27735 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/5562 [00:00<?, ?it/s]

done
Evaluating IM (noise=0.3, log_c_outcome)    ... 

...

replaying log with TBR, completed traces ::   0%|          | 0/5562 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/27735 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/5562 [00:00<?, ?it/s]

done

=== Quality Metrics (log_c_outcome) ===


,fitness,precision,generalization,simplicity_pm4py,size,cfc
"IM (noise=0.2, log_c_outcome)",0.9933,0.3325,0.9821,0.6691,93.0,41.0
"IM (noise=0.3, log_c_outcome)",0.9920,0.3397,0.9825,0.6593,89.0,40.0
"IM (noise=0.4, log_c_outcome)",0.9644,0.3613,0.9572,0.6348,73.0,34.0
"IM (noise=0.2, log_top30_denied)",0.9629,0.8666,0.9923,0.7971,55.0,16.0
"IM (noise=0.3, log_top30_denied)",0.9385,0.8746,0.9922,0.8246,47.0,12.0
"HM (dependency=0.2, log_top30_denied)",0.8905,0.9806,0.9908,0.7667,46.0,15.0


In [9]:
import os
os.makedirs("../report/figures", exist_ok=True)

pm4py.save_vis_petri_net(net_im02_c,      im_im02_c,      fm_im02_c,      "../report/figures/petri_im02_c.png")
pm4py.save_vis_petri_net(net_im03_c,      im_im03_c,      fm_im03_c,      "../report/figures/petri_im03_c.png")
pm4py.save_vis_petri_net(net_im04_c,      im_im04_c,      fm_im04_c,      "../report/figures/petri_im04_c.png")
pm4py.save_vis_petri_net(net_im02_denied, im_im02_denied, fm_im02_denied, "../report/figures/petri_im02_denied.png")
pm4py.save_vis_petri_net(net_im03_denied, im_im03_denied, fm_im03_denied, "../report/figures/petri_im03_denied.png")
pm4py.save_vis_petri_net(net_hm02,        im_hm02,        fm_hm02,        "../report/figures/petri_hm02.png")

pm4py.save_vis_bpmn(bpmn_im02_c,      "../report/figures/bpmn_final02_c.pdf")
pm4py.save_vis_bpmn(bpmn_im03_c,      "../report/figures/bpmn_final03_c.pdf")
pm4py.save_vis_bpmn(bpmn_im04_c,      "../report/figures/bpmn_final04_c.pdf")
pm4py.save_vis_bpmn(bpmn_im02_denied, "../report/figures/bpmn_final02_denied.pdf")
pm4py.save_vis_bpmn(bpmn_im03_denied, "../report/figures/bpmn_final03_denied.pdf")

print("Saved")

Saved


# Decision Mining Analysis


In [10]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
from pm4py.algo.conformance.tokenreplay import algorithm as tbr_algo
import warnings; warnings.filterwarnings('ignore')

NET, IM, FM = net_im02_denied, im_im02_denied, fm_im02_denied
LOG_EL = pm4py.convert_to_event_log(log_outcomes_original)

In [11]:
from sklearn.tree import DecisionTreeClassifier, export_text

drop_always = {CASE_COL, ACT_COL, TS_COL, 'outcome','Accepted','Selected',  
               'EventID', 'Action', 'EventOrigin', 'lifecycle:transition',
               '@@index', '@@case_index', 'start_timestamp', 'OfferID'}

df_sorted = log_c_outcome.sort_values([CASE_COL, TS_COL]).copy()
df_sorted['next_act'] = df_sorted.groupby(CASE_COL)[ACT_COL].shift(-1)

ff_cols = [c for c in df_sorted.columns if c not in {CASE_COL, ACT_COL, TS_COL, 'next_act'}]
df_sorted[ff_cols] = df_sorted.groupby(CASE_COL)[ff_cols].ffill()

GATEWAYS = [
    ("GW1: after A_Create Application",     "A_Create Application",       {"A_Submitted",            "A_Concept"},   {}),
    ("GW2: after A_Submitted",              "A_Submitted",                {"W_Handle leads",         "A_Concept"},   {}),
    ("GW3: after O_Sent (mail and online)", "O_Sent (mail and online)",   {"W_Complete application", "A_Complete"},  {}),
    ("GW4: after A_Complete",               "A_Complete",                 {"A_Validating",           "A_Cancelled"}, {}),
    ("GW5: after A_Incomplete",             "A_Incomplete",               {"A_Denied",           "O_Accepted"}, {}),
    ("GW6: after A_Incomplete",             "A_Incomplete",               {"A_Validating", "A_Denied", "O_Accepted"},
       {"A_Denied": "terminal", "O_Accepted": "terminal"}),
]

FORCE_CAT = {'case:LoanGoal', 'case:ApplicationType'}

def encode(df):
    df = df.copy()
    cat_cols = [c for c in df.columns
                if not pd.api.types.is_numeric_dtype(df[c]) or c in FORCE_CAT]
    num_cols = [c for c in df.columns if c not in cat_cols]
    df_num = df[num_cols].fillna(-1).astype(float)
    if cat_cols:
        df_cat = pd.get_dummies(df[cat_cols].astype(str), prefix_sep='=')
        return pd.concat([df_num, df_cat], axis=1)
    return df_num

gw_datasets = {}
for gw_label, trigger, branches, remap in GATEWAYS:
    trigger_rows = df_sorted[
        (df_sorted[ACT_COL] == trigger) &
        (df_sorted['next_act'].isin(branches))
    ].drop_duplicates(CASE_COL).set_index(CASE_COL)

    feat_cols = [c for c in trigger_rows.columns if c not in drop_always | {ACT_COL, 'next_act'}]
    X_raw = trigger_rows[feat_cols].copy()
    if 'org:resource' in trigger_rows.columns:
        X_raw['org:resource'] = trigger_rows['org:resource']
    y = trigger_rows['next_act']
    if remap:
        y = y.map(lambda x: remap.get(x, x))

    if y.nunique() < 2:
        print(f"Skipping {gw_label}: only one class")
        continue

    X = encode(X_raw)
    df_gw = X.copy()
    df_gw['_label'] = y
    gw_datasets[gw_label] = df_gw
    print(f"{gw_label}: {len(df_gw):,} cases — {y.value_counts().to_dict()}")

GW1: after A_Create Application: 31,411 cases — {'A_Submitted': 20339, 'A_Concept': 11072}
GW2: after A_Submitted: 20,321 cases — {'A_Concept': 16865, 'W_Handle leads': 3456}
GW3: after O_Sent (mail and online): 30,814 cases — {'W_Complete application': 18654, 'A_Complete': 12160}
GW4: after A_Complete: 26,341 cases — {'A_Validating': 18307, 'A_Cancelled': 8034}
GW5: after A_Incomplete: 4,970 cases — {'O_Accepted': 4781, 'A_Denied': 189}
GW6: after A_Incomplete: 12,368 cases — {'A_Validating': 9300, 'terminal': 3068}


In [12]:
gw_trees = {}
for gw_label, df_gw in gw_datasets.items():
    X = df_gw.drop(columns='_label')
    y = df_gw['_label']
    dt = DecisionTreeClassifier(max_depth=1, class_weight='balanced', random_state=42)
    dt.fit(X, y)
    gw_trees[gw_label] = (dt, X.columns.tolist(), y.unique().tolist())
    from sklearn.model_selection import cross_val_score
    from sklearn.metrics import accuracy_score
    acc_train = accuracy_score(y, dt.predict(X))
    cv_acc = cross_val_score(dt, X, y, cv=5, scoring='accuracy').mean()
    print(f"\n{'='*60}")
    print(f"Gateway: {gw_label}")
    print(f"Accuracy — train: {acc_train:.1%}  |  5-fold CV: {cv_acc:.1%}")
    print(export_text(dt, feature_names=X.columns.tolist()))



Gateway: GW1: after A_Create Application
Accuracy — train: 100.0%  |  5-fold CV: 100.0%
|--- org:resource=User_1 <= 0.50
|   |--- class: A_Concept
|--- org:resource=User_1 >  0.50
|   |--- class: A_Submitted


Gateway: GW2: after A_Submitted
Accuracy — train: 73.8%  |  5-fold CV: 73.8%
|--- case:LoanGoal=Existing loan takeover <= 0.50
|   |--- class: A_Concept
|--- case:LoanGoal=Existing loan takeover >  0.50
|   |--- class: W_Handle leads


Gateway: GW3: after O_Sent (mail and online)
Accuracy — train: 46.1%  |  5-fold CV: 46.1%
|--- case:RequestedAmount <= 4500.00
|   |--- class: W_Complete application
|--- case:RequestedAmount >  4500.00
|   |--- class: A_Complete


Gateway: GW4: after A_Complete
Accuracy — train: 72.8%  |  5-fold CV: 72.8%
|--- CreditScore <= 297.00
|   |--- class: A_Cancelled
|--- CreditScore >  297.00
|   |--- class: A_Validating


Gateway: GW5: after A_Incomplete
Accuracy — train: 92.0%  |  5-fold CV: 92.0%
|--- CreditScore <= 297.00
|   |--- class: A_Denied
|-